# Tuần 5 — Kiểm định detector R-peak

Notebook này đối sánh một-một R-peak tự động của tuần 3–4 với annotation `.atr` gốc của MIT-BIH trong dung sai ±50 ms. Mọi bảng và hình được tạo trực tiếp khi chạy notebook.

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Tìm thư mục dự án dù notebook được chạy từ VS Code hay Jupyter.
PROJECT_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mitdb').is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Không tìm thấy thư mục mitdb ở thư mục hiện tại hoặc thư mục cha.')
WEEK3_DIR = PROJECT_DIR / 'Tuần 3-4' / 'ket_qua_ecg'
RESULT_DIR = PROJECT_DIR / 'Tuần 5-6-7' / 'ket_qua_tuan_5'
FIG_DIR = RESULT_DIR / 'figures'
DATA_DIR = PROJECT_DIR / 'mitdb' / 'mit-bih-arrhythmia-database-1.0.0' / 'mit-bih-arrhythmia-database-1.0.0'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

with (WEEK3_DIR / 'manifest.json').open(encoding='utf-8') as f:
    manifest = json.load(f)
RECORDS = manifest['records']
TOLERANCE_MS = 50
# Chỉ các annotation biểu diễn nhịp được dùng làm mốc QRS chuẩn.
BEAT_SYMBOLS = set('N L R e j A a J S V E F / f Q ?'.split())

def match_one_to_one(detected, reference, symbols, tolerance_samples):
    """Ghép hai dãy tăng dần; mỗi peak/annotation chỉ được dùng tối đa một lần."""
    detected = np.asarray(detected, dtype=int)
    reference = np.asarray(reference, dtype=int)
    i = j = 0
    matches, false_positives, false_negatives = [], [], []
    while i < len(detected) and j < len(reference):
        delta = int(detected[i] - reference[j])
        if abs(delta) <= tolerance_samples:
            matches.append((int(detected[i]), int(reference[j]), symbols[j], delta))
            i += 1
            j += 1
        elif detected[i] < reference[j]:
            false_positives.append(int(detected[i]))
            i += 1
        else:
            false_negatives.append((int(reference[j]), symbols[j]))
            j += 1
    false_positives.extend(map(int, detected[i:]))
    false_negatives.extend((int(reference[k]), symbols[k]) for k in range(j, len(reference)))
    return matches, false_positives, false_negatives

metric_rows, match_rows, fp_rows, fn_rows = [], [], [], []
examples = {}
for record in RECORDS:
    saved = np.load(WEEK3_DIR / record / 'beats_180.npz')
    detected = saved['r_peaks']
    fs = float(saved['fs'])
    annotation = wfdb.rdann(str(DATA_DIR / record), 'atr')
    keep = np.array([symbol in BEAT_SYMBOLS for symbol in annotation.symbol], dtype=bool)
    reference = np.asarray(annotation.sample, dtype=int)[keep]
    symbols = np.asarray(annotation.symbol, dtype=object)[keep]
    tolerance_samples = int(round(TOLERANCE_MS / 1000 * fs))
    matches, false_positives, false_negatives = match_one_to_one(detected, reference, symbols, tolerance_samples)
    tp, fp, fn = len(matches), len(false_positives), len(false_negatives)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    errors = [row[3] / fs * 1000 for row in matches]
    metric_rows.append({'record': record, 'fs_hz': fs, 'tolerance_ms': TOLERANCE_MS,
                        'reference_beats': len(reference), 'detected_peaks': len(detected),
                        'TP': tp, 'FP': fp, 'FN': fn, 'precision_ppv': precision,
                        'recall_sensitivity': recall, 'f1': f1,
                        'mean_abs_error_ms': float(np.mean(np.abs(errors))) if errors else np.nan})
    match_rows.extend({'record': record, 'detected_sample': d, 'reference_sample': r, 'symbol': s,
                       'error_samples': delta, 'error_ms': delta / fs * 1000}
                      for d, r, s, delta in matches)
    fp_rows.extend({'record': record, 'detected_sample': peak} for peak in false_positives)
    fn_rows.extend({'record': record, 'reference_sample': peak, 'symbol': symbol}
                   for peak, symbol in false_negatives)
    if matches and 'TP' not in examples:
        examples['TP'] = (record, matches[0][0], matches[0][1])
    if false_positives and 'FP' not in examples:
        examples['FP'] = (record, false_positives[0], None)
    if false_negatives and 'FN' not in examples:
        examples['FN'] = (record, None, false_negatives[0][0])

metrics = pd.DataFrame(metric_rows).sort_values('record')
matches_df = pd.DataFrame(match_rows)
fp_df = pd.DataFrame(fp_rows)
fn_df = pd.DataFrame(fn_rows)
total_tp, total_fp, total_fn = metrics[['TP', 'FP', 'FN']].sum()
overall_precision = total_tp / (total_tp + total_fp)
overall_recall = total_tp / (total_tp + total_fn)
overall_f1 = 2 * overall_precision * overall_recall / (overall_precision + overall_recall)
summary = {'records': int(len(metrics)), 'tolerance_ms': TOLERANCE_MS, 'TP': int(total_tp),
           'FP': int(total_fp), 'FN': int(total_fn), 'precision_ppv': overall_precision,
           'recall_sensitivity': overall_recall, 'f1': overall_f1}
print('KẾT QUẢ TOÀN BỘ 48 RECORD')
print(pd.DataFrame([summary]).round(4).to_string(index=False))
print(metrics.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(metrics['record'], metrics['precision_ppv'], label='Precision/PPV', alpha=.8)
ax.plot(metrics['record'], metrics['recall_sensitivity'], color='crimson', marker='o', label='Recall/Sensitivity')
ax.axhline(overall_precision, color='tab:blue', ls='--', lw=1, label=f'PPV toàn bộ = {overall_precision:.3f}')
ax.axhline(overall_recall, color='crimson', ls='--', lw=1, label=f'Recall toàn bộ = {overall_recall:.3f}')
ax.set_ylim(0, 1.05); ax.set_xlabel('Record'); ax.set_ylabel('Điểm số')
ax.set_title('Precision/Recall detector R-peak theo record (±50 ms)')
ax.tick_params(axis='x', rotation=90); ax.grid(axis='y', alpha=.25); ax.legend(ncol=2)
fig.tight_layout(); fig.savefig(FIG_DIR / 'precision_recall_theo_record.png', dpi=180); plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(matches_df['error_ms'], bins=41, color='tab:purple', edgecolor='white')
ax.axvline(0, color='black', lw=1); ax.set_xlabel('Sai lệch detector - annotation (ms)')
ax.set_ylabel('Số nhịp'); ax.set_title('Phân bố sai lệch vị trí R của các ghép đúng')
fig.tight_layout(); fig.savefig(FIG_DIR / 'histogram_sai_lech_R_peak.png', dpi=180); plt.show()

fig, axes = plt.subplots(len(examples), 1, figsize=(12, 3.2 * len(examples)), squeeze=False)
for ax, (kind, (record, detected_peak, reference_peak)) in zip(axes[:, 0], examples.items()):
    signal = np.load(WEEK3_DIR / record / 'cleaned.npz')['filtered']
    center = detected_peak if detected_peak is not None else reference_peak
    left, right = max(0, center - 360), min(len(signal), center + 361)
    time = np.arange(left, right) / 360
    ax.plot(time, signal[left:right], color='tab:blue', lw=.9, label='ECG đã lọc')
    if detected_peak is not None: ax.axvline(detected_peak / 360, color='crimson', ls='--', label='R detector')
    if reference_peak is not None: ax.axvline(reference_peak / 360, color='black', ls=':', label='Annotation chuẩn')
    ax.set_title(f'Ví dụ {kind} — record {record}')
    ax.set_xlabel('Thời gian (giây)'); ax.set_ylabel('mV'); ax.grid(alpha=.25); ax.legend()
fig.tight_layout(); fig.savefig(FIG_DIR / 'vi_du_TP_FP_FN.png', dpi=180); plt.show()

metrics.to_csv(RESULT_DIR / 'detector_metrics_by_record.csv', index=False, encoding='utf-8-sig')
matches_df.to_csv(RESULT_DIR / 'matched_peaks.csv', index=False, encoding='utf-8-sig')
fp_df.to_csv(RESULT_DIR / 'false_positive_peaks.csv', index=False, encoding='utf-8-sig')
fn_df.to_csv(RESULT_DIR / 'false_negative_beats.csv', index=False, encoding='utf-8-sig')
(RESULT_DIR / 'detector_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Đã lưu kết quả vào: {RESULT_DIR}')
